In [22]:
import pandas as pd
import seaborn as sns
import plotly.express as px
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression

In [23]:
df_path = "C:/Users/nicow/OneDrive/01_job/Technical_Steps/estuaire/data_challenge.csv"
df = pd.read_csv(df_path, sep=";")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df["Take-off Time (UTC)"] = pd.to_datetime(
    df["Take-off Time (UTC)"], format="%d/%m/%Y %H:%M"
)
df

,Aircraft,Engine,Role,Registration,Seats,Cargo Capacity (t),Flight Number,Date,Take-off Time (UTC),Origin Airport,Destination Airport,Distance OD (km),Distance Flown (km),CO2 (kgCO2e),Contrails (kgCO2e)
0,A20N,PW1127G-JM,PAX,EC-NFH,186.0,0.0,VY7835,2023-06-15,2023-06-15 16:01:00,EGKK,LEBL,1109.053917,1155.727519,11253.969499,0.000000
1,A20N,PW1127G-JM,PAX,EC-NIJ,186.0,0.0,VY7830,2023-06-15,2023-06-15 05:12:00,LEBL,EGKK,1109.053917,1255.367953,11965.946434,16370.295266
2,A20N,PW1127G-JM,PAX,EC-NCT,186.0,0.0,VY3212,2023-12-31,2023-12-31 10:52:00,LEBL,GCXO,2194.502853,NaN,19808.910939,NaN
3,A20N,PW1127G-JM,PAX,EC-NCF,186.0,0.0,VY6213,2023-10-09,2023-10-09 15:49:00,LEMG,LIRF,1548.024410,1590.238353,14352.387644,41217.801880
4,A321,CFM56-5B3/3,PAX,EC-NLY,220.0,0.0,VY8748,2023-11-02,2023-11-02 13:50:00,EGCC,LEBL,1379.156221,1492.741633,20147.168918,11686.423628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,A321,V2533-A5,PAX,EC-MOO,220.0,0.0,VY6937,2023-07-28,2023-07-28 13:44:00,LEIB,LFPO,1098.264174,1234.678762,17236.971394,0.000000
19996,A320,V2527-A5,PAX,EC-MEQ,186.0,0.0,VY3293,2023-09-04,2023-09-04 17:18:00,EKCH,EHVK,634.882463,NaN,10097.686470,NaN
19997,A321,V2533-A5,PAX,EC-MQL,220.0,0.0,VY1883,2023-02-13,2023-02-13 20:41:00,EDDB,LEBL,1502.412483,1568.138924,21013.515601,0.000000
19998,A20N,PW1127G-JM,PAX,EC-NDC,186.0,0.0,VY2112,2023-02-25,2023-02-25 18:39:00,LEMG,LEBL,765.966390,876.445651,9263.071031,0.000000


In [24]:
# drop useless columns
df = df.drop(columns=["Cargo Capacity (t)", "Role"])


# count nan values per column
print(df.isna().sum())
df = df.dropna()

Aircraft                 0
Engine                   0
Registration             0
Seats                    0
Flight Number            0
Date                     0
Take-off Time (UTC)      0
Origin Airport           0
Destination Airport      0
Distance OD (km)         1
Distance Flown (km)    671
CO2 (kgCO2e)             1
Contrails (kgCO2e)     671
dtype: int64


In [25]:
px.histogram(df, x="Contrails (kgCO2e)")

In [26]:
px.histogram(df.where(df["Contrails (kgCO2e)"] != 0), x="Contrails (kgCO2e)")

In [27]:
# count rows where Contrails (kgCO2e) is 0, and get rid of them
print(df[df["Contrails (kgCO2e)"] == 0].shape[0])
print(len(df))
df = df[df["Contrails (kgCO2e)"] != 0]
print(len(df))

12344
19329
6985


In [28]:
# # Calcul de la date minimale
# min_date = df['Date'].min()

# # Conversion des colonnes en minutes depuis la date minimale
# df['Date (minutes)'] = (df['Date'] - min_date).dt.total_seconds() / 60
# df['Take-off Time (minutes)'] = (df['Take-off Time (UTC)'] - min_date).dt.total_seconds() / 60

In [29]:
categorical_features = [
    "Aircraft",
    "Engine",
    "Registration",
    "Flight Number",
    "Origin Airport",
    "Destination Airport",
]
numerical_features = df.columns.difference(categorical_features)


encoder = OrdinalEncoder()
categorical_data = encoder.fit_transform(df[categorical_features])


categorical_data = pd.DataFrame(categorical_data, columns=categorical_features).dropna()


data = pd.concat([df[numerical_features], categorical_data], axis=1).dropna()
data

,CO2 (kgCO2e),Contrails (kgCO2e),Date,Distance Flown (km),Distance OD (km),Seats,Take-off Time (UTC),Aircraft,Engine,Registration,Flight Number,Origin Airport,Destination Airport
1,11965.946434,16370.295266,2023-06-15,1255.367953,1109.053917,186.0,2023-06-15 05:12:00,0.0,5.0,103.0,544.0,46.0,94.0
3,14352.387644,41217.801880,2023-10-09,1590.238353,1548.024410,186.0,2023-10-09 15:49:00,3.0,7.0,46.0,248.0,59.0,47.0
4,20147.168918,11686.423628,2023-11-02,1492.741633,1379.156221,220.0,2023-11-02 13:50:00,3.0,1.0,21.0,565.0,66.0,94.0
10,10632.932103,434360.293772,2023-02-13,787.970150,701.154153,180.0,2023-02-13 20:38:00,3.0,1.0,22.0,531.0,96.0,48.0
13,14932.163157,1518.480424,2023-04-28,1247.983036,1088.665991,180.0,2023-04-28 07:20:00,3.0,1.0,18.0,929.0,40.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6951,7061.216434,61717.662191,2023-01-02,566.307141,453.468536,186.0,2023-01-02 21:13:00,0.0,5.0,107.0,188.0,42.0,48.0
6965,16399.772151,537096.819247,2023-10-01,1405.901790,1241.136244,180.0,2023-10-01 16:30:00,0.0,5.0,118.0,134.0,5.0,48.0
6969,20146.491584,15767.109699,2023-03-22,1807.618967,1645.205998,180.0,2023-03-22 06:02:00,3.0,7.0,42.0,47.0,17.0,44.0
6974,8543.747535,5226.820196,2023-11-29,563.371344,453.468536,180.0,2023-11-29 15:29:00,0.0,5.0,108.0,51.0,40.0,47.0


In [30]:
data.describe()

,CO2 (kgCO2e),Contrails (kgCO2e),Date,Distance Flown (km),Distance OD (km),Seats,Take-off Time (UTC),Aircraft,Engine,Registration,Flight Number,Origin Airport,Destination Airport
count,2407.000000,2.407000e+03,2407,2407.000000,2407.000000,2407.000000,2407,2407.000000,2407.000000,2407.000000,2407.000000,2407.000000,2407.000000
mean,15746.860923,4.108165e+04,2023-06-29 00:32:18.346489344,1361.944551,1255.274886,188.009971,2023-06-29 13:07:35.471541248,2.523473,4.874533,64.118405,464.562111,45.444121,52.022850
min,7061.216434,-7.379588e+05,2023-01-01 00:00:00,456.090348,0.000000,144.000000,2023-01-01 05:19:00,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000
25%,12100.541097,-1.635407e+03,2023-04-09 00:00:00,958.778237,854.821215,180.000000,2023-04-09 07:14:30,2.000000,2.000000,33.000000,173.000000,35.000000,36.000000
50%,14780.329054,5.309812e+02,2023-06-09 00:00:00,1268.656798,1109.053917,186.000000,2023-06-09 04:59:00,3.000000,5.000000,65.000000,474.000000,40.000000,48.000000
75%,17953.711087,2.732005e+04,2023-10-04 00:00:00,1603.627850,1485.689825,186.000000,2023-10-04 07:14:30,3.000000,7.000000,94.000000,757.000000,59.000000,68.000000
max,43357.136145,2.150571e+06,2023-12-31 00:00:00,4151.327430,3849.304846,230.000000,2023-12-31 17:22:00,4.000000,8.000000,125.000000,972.000000,97.000000,109.000000
std,5185.960909,1.520994e+05,NaN,529.363474,518.541106,18.068676,NaN,1.313721,2.581998,35.540999,308.793576,22.853488,24.883834


In [31]:
# sns.pairplot(data)

In [32]:
data_pre_pca = data.drop(columns=["Contrails (kgCO2e)"])
n_components = len(data_pre_pca.columns)
pca = PCA(n_components=n_components)


x_norm = ((data_pre_pca - data_pre_pca.mean()) / data_pre_pca.std()).values


data_pca = pca.fit_transform(x_norm)



data["PC1"] = data_pca[:, 0]


data["PC2"] = data_pca[:, 1]


data["PC3"] = data_pca[:, 2]



# 3D PCA plot


fig0 = px.scatter_3d(
    data,

    x="PC1",
    y="PC2",
    z="PC3",
    height=800,
    width=800,
    color="Contrails (kgCO2e)",
    color_continuous_scale=px.colors.diverging.Portland,
).update_traces(marker=dict(size=2))


fig0.show()


# 2D PCA plot


fig1 = px.scatter(
    data,

    x="PC1",
    y="PC2",
    color="Contrails (kgCO2e)",
    color_continuous_scale=px.colors.diverging.Portland,
)



loadings = pca.components_.T * np.sqrt(pca.explained_variance_)



for i, feature in enumerate(data_pre_pca.columns[:-3]):

    print(feature)

    fig1.add_annotation(
        ax=0,
        ay=0,
        axref="x",
        ayref="y",
        x=loadings[i, 0],
        y=loadings[i, 1],
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top",
    )


    fig1.add_annotation(
        x=loadings[i, 0],
        y=loadings[i, 1],
        ax=0,
        ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )



fig1.show()

C:\Users\nicow\AppData\Local\Temp\ipykernel_21908\3531610547.py:6: PerformanceWarning:

Adding/subtracting object-dtype array to DatetimeArray not vectorized.

C:\Users\nicow\AppData\Local\Temp\ipykernel_21908\3531610547.py:6: PerformanceWarning:

Adding/subtracting object-dtype array to DatetimeArray not vectorized.



CO2 (kgCO2e)
Date
Distance Flown (km)
Distance OD (km)
Seats
Take-off Time (UTC)
Aircraft
Engine
Registration


# PLS

In [33]:
correlation = data.corr()
correlation["Contrails (kgCO2e)"].apply(lambda x: abs(x)).sort_values(
    ascending=False
).head(12)

Contrails (kgCO2e)     1.000000
CO2 (kgCO2e)           0.123096
Distance Flown (km)    0.112556
PC1                    0.111199
Distance OD (km)       0.084733
Seats                  0.062356
Aircraft               0.018799
Destination Airport    0.018436
Take-off Time (UTC)    0.017095
Engine                 0.016981
Date                   0.016506
Flight Number          0.011081
Name: Contrails (kgCO2e), dtype: float64

Right to use CO2?

In [34]:
feature_selection_list = [
    "Distance Flown (km)",
    "Distance OD (km)",
    "Seats",
    "Aircraft",
    "Destination Airport",
    "Take-off Time (UTC)",
    "Engine",
    "Date",
]

In [35]:
data["Contrails (kgCO2e)"].describe()

count    2.407000e+03
mean     4.108165e+04
std      1.520994e+05
min     -7.379588e+05
25%     -1.635407e+03
50%      5.309812e+02
75%      2.732005e+04
max      2.150571e+06
Name: Contrails (kgCO2e), dtype: float64

In [36]:
y = data["Contrails (kgCO2e)"]
# normalize y and apply log transformation
y = np.log(y - y.min() + 1)

pls = PLSRegression(n_components=3)
pls.fit(x_norm, y)

data["PLS_PC1"] = pls.x_scores_[:, 0]
data["PLS_PC2"] = pls.x_scores_[:, 1]
data["PLS_PC3"] = pls.x_scores_[:, 2]

# plotting the loadings
loadings = pls.x_loadings_
fig = px.scatter(
    data,
    x="PLS_PC1",
    y="PLS_PC2",
    color="Contrails (kgCO2e)",
    color_continuous_scale=px.colors.diverging.Portland,
)

for i, feature in enumerate(feature_selection_list):
    fig.add_annotation(
        ax=0,
        ay=0,
        axref="x",
        ayref="y",
        x=loadings[i, 0],
        y=loadings[i, 1],
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top",
    )
    fig.add_annotation(
        x=loadings[i, 0],
        y=loadings[i, 1],
        ax=0,
        ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [37]:
# linear regression y = f(PLS_PC1)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


X = data[["PLS_PC1"]].values
y = data["Contrails (kgCO2e)"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
reg = LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X)
fig = px.scatter(
    data,
    x="PLS_PC1",
    y="Contrails (kgCO2e)",
    color="Contrails (kgCO2e)",
    color_continuous_scale=px.colors.diverging.Portland,
)
fig.add_scatter(
    x=data["PLS_PC1"],
    y=y_pred,
    mode="lines",
    line=dict(color="red", width=2),
    name="Regression Line",
)
fig.update_layout(
    title="Linear Regression of y = f(PLS_PC1)",
    xaxis_title="PLS_PC1",
    yaxis_title="Contrails (kgCO2e)",
)
fig.show()
# print r squared and mean squared error
print("R^2: ", reg.score(X_test, y_test))
print("Mean Squared Error: ", np.mean((y_pred - y) ** 2))

R^2:  -0.011013782902183245
Mean Squared Error:  22836833855.507996


# XGBoost

In [38]:
input_features = [
    "Date",
    "Distance Flown (km)",
    "Distance OD (km)",
    "Seats",
    "Take-off Time (UTC)",
    "Aircraft",
    "Engine",
    "Registration",
    "Flight Number",
    "Origin Airport",
    "Destination Airport",
]

# Calcul de la date minimale
min_date = data["Date"].min()

# Conversion des colonnes en minutes depuis la date minimale
data["Date"] = (data["Date"] - min_date).dt.total_seconds() / 3600 / 24
data["Take-off Time (UTC)"] = (
    (data["Take-off Time (UTC)"] - min_date).dt.total_seconds() / 3600 / 24
)

In [39]:
# using xgboost for regression on Contrails (kgCO2e), using data[feature_selection_list] as features
# we use cross validation to find the best parameters
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint


X = data[input_features].values
y = data["Contrails (kgCO2e)"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# define the model
model = XGBRegressor()
# define the parameter grid for RandomizedSearchCV
param_dist = {
    "n_estimators": randint(50, 200),
    "max_depth": randint(2, 10),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.5, 0.5),
    "gamma": uniform(0, 5),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(0, 1),
}
# perform Randomized search with cross validation

grid_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=1,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
# print best parameters
print("Best parameters: ", grid_search.best_params_)
# print best score
print("Best score: ", grid_search.best_score_)
# fit the model with the best parameters
model = grid_search.best_estimator_
model.fit(X_train, y_train)
# make predictions on the test set
y_pred = model.predict(X_test)
# calculate the mean squared error and r squared
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("Mean squared error: ", mse)
print("Root mean squared error: ", rmse)
print("R squared: ", r2)

# plot feature importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
fig = px.bar(
    y=importances[indices],
    x=data[input_features].columns[indices],
    labels={"x": "Feature", "y": "Importance"},
    title="Feature Importance",
)
fig.update_traces(marker_color="blue")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:  {'colsample_bytree': np.float64(0.9495092696004115), 'gamma': np.float64(1.6788262954735038), 'max_depth': 2, 'min_child_weight': 8, 'n_estimators': 83, 'reg_alpha': np.float64(0.3733553898090237), 'reg_lambda': np.float64(0.46062215638215065), 'subsample': np.float64(0.7855460746120042)}
Best score:  -24307368318.524372
Mean squared error:  21622068108.170048
Root mean squared error:  147044.44262932907
R squared:  -0.08516413720452798


In [40]:
# save the model
import joblib

joblib.dump(model, "trained_models/xgboost_model.pkl")

['trained_models/xgboost_model.pkl']

In [41]:
# Same with random forest regressor
from sklearn.ensemble import RandomForestRegressor

# define the model
model = RandomForestRegressor()
# define the parameter grid for RandomizedSearchCV
param_dist = {
    "n_estimators": randint(50, 200),
    "max_depth": randint(2, 10),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["auto", "sqrt"],
}
# perform Randomized search with cross validation
grid_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=1,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best parameters: ", grid_search.best_params_)
print("Best score: ", grid_search.best_score_)
# fit the model with the best parameters
model = grid_search.best_estimator_
model.fit(X_train, y_train)
# make predictions on the test set
y_pred = model.predict(X_test)
# calculate the mean squared error and r squared
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("Mean squared error: ", mse)
print("Root mean squared error: ", rmse)
print("R squared: ", r2)
# plot feature importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
fig = px.bar(
    y=importances[indices],
    x=data[input_features].columns[indices],
    labels={"x": "Feature", "y": "Importance"},
    title="Feature Importance",
)
fig.update_traces(marker_color="blue")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Fitting 5 folds for each of 10 candidates, totalling 50 fits


c:\Users\nicow\OneDrive\01_job\Technical_Steps\estuaire\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning:


35 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
35 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\nicow\OneDrive\01_job\Technical_Steps\estuaire\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\nicow\OneDrive\01_job\Technical_Steps\estuaire\.venv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
  File "c:\Users\nicow\OneDrive\01_job\Technical_Steps\estuaire\.venv\Lib\

Best parameters:  {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 6, 'min_samples_split': 3, 'n_estimators': 146}
Best score:  -23307098953.026054
Mean squared error:  20432356126.88225
Root mean squared error:  142941.79279301857
R squared:  -0.025455104320285526


# Neural Network Regressor

In [ ]:
from nn_model import NNRegressor
from torch.utils.data import DataLoader, TensorDataset
import torch
from sklearn.model_selection import train_test_split

n_dimension = len(data[input_features].columns)
model = NNRegressor(nb_dimensions=n_dimension)
X = data[input_features].values
X_norm = ((X - X.mean()) / X.std()).astype(np.float32)
y = data["Contrails (kgCO2e)"].values
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42
)
tensor_x_train = torch.tensor(X_train, dtype=torch.float32)
tensor_y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
train_dataset = TensorDataset(tensor_x_train, tensor_y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
tensor_x_test = torch.tensor(X_test, dtype=torch.float32)
tensor_y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
test_dataset = TensorDataset(tensor_x_test, tensor_y_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

n_epochs = 1000
train_loss_list = []
test_loss_list = []
for epoch in range(n_epochs):
    train_loss = model.train(train_dataloader)
    test_loss = model.test(test_dataloader)
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    print(f"Epoch {epoch+1}, Train loss: {train_loss}, Test loss: {test_loss}")

# plot the train and test loss
px.scatter(x=range(n_epochs), y=train_loss_list, title="Train loss").add_scatter(
    y=test_loss_list, name="Test loss", mode="markers"
).show()

# make predictions on the test set and print rmse and r squared
y_pred = model.forward(tensor_x_test).detach().numpy()
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("Mean squared error: ", mse)
print("Root mean squared error: ", rmse)
print("R squared: ", r2)

Epoch 1, Train loss: 25199102730.491802, Test loss: 22998135008.0
Epoch 2, Train loss: 25011742069.5082, Test loss: 22317061536.0
Epoch 3, Train loss: 24311700219.803265, Test loss: 20810066464.0
Epoch 4, Train loss: 23487703474.36066, Test loss: 20450026176.0
Epoch 5, Train loss: 23415913864.39344, Test loss: 20751974976.0
Epoch 6, Train loss: 27849729760.524582, Test loss: 20404342976.0
Epoch 7, Train loss: 23340306289.311474, Test loss: 20583793120.0
Epoch 8, Train loss: 24032948096.000004, Test loss: 20356612704.0
Epoch 9, Train loss: 23338025876.983604, Test loss: 20596486656.0
Epoch 10, Train loss: 23402285230.163925, Test loss: 20460273920.0
Epoch 11, Train loss: 23321898439.34427, Test loss: 20690150976.0
Epoch 12, Train loss: 24287783251.93443, Test loss: 20673471680.0
Epoch 13, Train loss: 23561552975.7377, Test loss: 20498375488.0
Epoch 14, Train loss: 24370864795.278698, Test loss: 20401031648.0
Epoch 15, Train loss: 23307744730.2295, Test loss: 20542264768.0
Epoch 16, Trai

Mean squared error:  20223530206.41655
Root mean squared error:  142209.45892034238
R squared:  -0.014974589751816314


In [ ]:
# save the model
import joblib

joblib.dump(model, "trained_models/nn_model.pkl")